In [ ]:
%pylab inline
from scipy.integrate import solve_ivp
import tqdm, time
import types

In [ ]:
# Virus parmeters for COVID19
# Median R0 from https://academic.oup.com/jtm/article/27/2/taaa021/5735319
R0 = 2.4
R0_q = 0.8 # R0 under quarantine
γ = 1/14 # COVID19 recovery time 2 weeks
[β, β_q] = array([R0, R0_q]) * γ

Hc = 8/1000 # Hospital capicity
Hrate = 7.3/100 # % infected that need hospitalization
Mlow = 1/100 # Mortality rate before hospital capicity is exceeded
Mhigh = 2*Mlow # Mortality rate after hospital capicity is exeeded

P0 = 100

In [ ]:
# Quarantine strategies. Returns whether quarantined or not
def q_threshold( t, S, I, R ):
    P = S + I + R
    q_threshold.Q = (I/P > q_threshold.ilow) if q_threshold.Q \
        else (I/P > q_threshold.ihigh)
    return q_threshold.Q
q_threshold.Q = False
q_threshold.ihigh = Hc/Hrate
q_threshold.ilow = .9 * q_threshold.ihigh

def q_intermittent( t, S, I, R ):
    P = S + I + R
    if q_intermittent.Q:
        if t > q_intermittent.T + q_intermittent.len:
            q_intermittent.Q = False
    else:
        if I/P > q_intermittent.ihigh:
            q_intermittent.Q = True
            q_intermittent.T = t
    return q_intermittent.Q
q_intermittent.T = 0 # Time last quarrantine started
q_intermittent.Q = False
q_intermittent.ihigh = Hc/Hrate    # Level to start quarrantine at
q_intermittent.len = 2  # Length of quarrantine to impose
    

def q_interval( t, S, I, R ):
    return (q_interval.start <= t < q_interval.end)
q_interval.start=0
q_interval.end=30

In [ ]:
def SIR( t, y):
    (S, I, R, D, Tq) = y
    P = S + I + R # Total number of people alive
    
    Q = SIR.quarantine_status( t, S, I, R )
            
    β_now = β_q if Q else β    
    Mrate = Mlow if Hrate*I/P < Hc else Mhigh
    
    dS = -β_now * S * I / P
    dI = +β_now * S * I / P - γ * I
    dR = (1-Mrate) * γ * I
    dD = Mrate * γ * I
    dTq = float(Q)
    
    return array( [dS, dI, dR, dD, dTq] )
SIR.quarantine_status = lambda t, S, I, R: False

In [ ]:
# 30,000 infected in US on 3/23.
I0 = 30e3/300e6 * P0
#I0 = 1000 / 23.6e6 * P0 # Roughly 1000 cases in NYC on 3/16
R_0 = 0
S0 = P0 - I0 - R_0
Tq0 = 0
D0=0
T_max = 18*30 # 6 months

In [ ]:
# Test run (one solution)
q_intermittent.Q = False
q_intermittent.len = 25
q_intermittent.ihigh = Hc/Hrate

q_interval.start = 750
q_interval.end = q_interval.start + 30

SIR.quarantine_status = q_interval

sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=.1, dense_output=True )
(S, I, R, D, Tq) = sol.y
tt = sol.t

figsize( 12, 4 )
subplot( 1, 2, 1 )
plot( tt, S, label='$S$')
plot( tt, I, label='$I$')
plot( tt, R, label='$R$')
#plot( tt, Tq, label='$T_q$')

#plot( tt, full_like(tt, q_threshold.ilow*P ), 'C9--', label='Q0' )
#plot( tt, full_like(tt, q_threshold.ihigh*P ), 'C9--', label='Q1' )

plot( tt, full_like(tt, Hc*P0/Hrate), 'C8--', alpha=.5, label='Hc')

#plot( tt, S+I+R, 'C4--', label='P' )

#plot( sol.t, sol.y[2], label='$S_3$')
#plot( sol.t, sol.y[1], label='$I$')
#ylim(0, 500)
legend()

subplot( 1, 2, 2)
plot( tt, D, label='$D$')
#plot( tt, I/10, 'C1--', label='I')
plot( tt, R*Mlow, 'C2--', label='R')

title( 'Death % vs time')
legend()

In [ ]:
print( f'{Tq[-1]:.1f} days in quarrantine.',
     f'Total deaths: {D[-1]/P0*100:.3f}%',
     f'Infection peak: {tt[argmax(I)]:.1f}',
     f'Final R: {R[-1]/P0*100:.3f}' )

In [ ]:
period = types.SimpleNamespace()
#period.length = 30
period.length = 45

In [ ]:
# Comparisons for fixed period  quarantines
# Find level at which intermittent 3 day quarantine is roughly the right lenght
q_intermittent.Q = False
q_intermittent.len = 3
if period.length == 30:
    q_intermittent.ihigh = .095
elif period.length == 45:
    q_intermittent.ihigh = .075
else:
    raise Exception('Undefined period length')
SIR.quarantine_status = q_intermittent
sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=.01, dense_output=True )
(S, I, R, D, Tq) = sol.y
tt = sol.t
print( f'{Tq[-1]:.1f} days in quarrantine. Total deaths: {D[-1]/P0*100:.3f}%' )

period.intermittent=types.SimpleNamespace(
    ihigh=q_intermittent.ihigh, len=q_intermittent.len, qtime=Tq[-1],
    tt=tt, S=S, I=I, R=R, D=D, Tq=Tq )

In [ ]:
# Find when to start a contiguous quarantine of a month to minimzie mortality
SIR.quarantine_status = q_interval
q_start_range = arange( 65, 85, step=1.0 )
final_mortality = empty_like(q_start_range)
period.interval = types.SimpleNamespace()

for i, q_start in enumerate( tqdm.tqdm_notebook( q_start_range, total=len(q_start_range) )):
    q_interval.start = q_start
    q_interval.end = q_interval.start + period.length
    
    sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=.1, dense_output=True )
    (S, I, R, D, Tq) = sol.y
    tt = sol.t
    final_mortality[i] = D[-1]
    
    if not hasattr( period.interval, 'D' ) or final_mortality[i] < period.interval.D[-1]:
        # Min attained
        period.interval = types.SimpleNamespace(
            start=q_interval.start, end=q_interval.end, qtime=Tq[-1],
            S=S, I=I, R=R, D=D, tt=tt )

In [ ]:
figsize( 6, 4 )
plot( q_start_range, final_mortality, '.-' )
title( f'Min attained by starting on day {period.interval.start}')

In [ ]:
# One month quarantine imposed at the start
q_interval.start = 0
q_interval.end = q_interval.start + period.length
SIR.quarantine_status = q_interval
sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
             max_step=.1, dense_output=True )
(S, I, R, D, Tq) = sol.y
tt = sol.t

period.atstart = types.SimpleNamespace(
    start=q_interval.start, end=q_interval.end, qtime=Tq[-1],
    tt=tt, S=S, I=I, R=R, D=D, Tq=Tq )

In [ ]:
# Control (no quarantine)
q_interval.start = 0
q_interval.end = 0
SIR.quarantine_status = q_interval
sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
             max_step=.1, dense_output=True )
(S, I, R, D, Tq) = sol.y
tt = sol.t

period.control = types.SimpleNamespace(
    start=q_interval.start, end=q_interval.end, qtime=Tq[-1],
    tt=tt, S=S, I=I, R=R, D=D, Tq=Tq )

In [ ]:
# Comparison plot of infections
plot( period.atstart.tt, period.atstart.I, label='Quarantine imposed initially')
plot( period.interval.tt, period.interval.I, 
     label=f'Quarantine delayed to day {period.interval.start:.0f}')
plot( period.intermittent.tt, period.intermittent.I,
     label='Intermittent quarantine' )
plot( period.control.tt, period.control.I, 'C8--', alpha=.5,
     label='Control (no quarantine)' )
plot( period.interval.tt, full_like( period.interval.tt, Hc/Hrate*P0),
     'C9--', alpha=.5, label='Hospital capicity' )

xlabel( 'Time (days)' )
ylabel( '% of population infected' )
legend()

In [ ]:
# Comparison plot of cumulative mortalities
plot( period.atstart.tt, period.atstart.D, label='Quarantine imposed initially')
plot( period.interval.tt, period.interval.D, 
     label=f'Quarantine delayed to day {period.interval.start:.0f}')
plot( period.intermittent.tt, period.intermittent.D,
     label='Intermittent quarantine' )

plot( period.control.tt, period.control.D, 'C8--', alpha=.5,
     label='Control (no quarantine)' )

xlabel( 'Time (days)' )
ylabel( '% fatalities' )
legend()

In [ ]:
print( f'Total mortalities: Control: {period.control.D[-1]:.2f}, At start: {period.atstart.D[-1]:.2f}',
      f'Delayed: {period.interval.D[-1]:.2f}', f'Intermittent: {period.intermittent.D[-1]:.2f}' )

### Total mortalities vs length of quarantine

In [ ]:
Mhigh = Mlow      # Mortalities same when health care capicity is exceeded.
#Mhigh = 2*Mlow    # Mortalities doubled when health care capicity is exceeded.

In [ ]:
threshold = types.SimpleNamespace()
ihigh_range = logspace( -2, log10(.2), num=30 )
total_qtime = empty_like(ihigh_range)
final_mortality = empty_like(ihigh_range)

SIR.quarantine_status = q_threshold
for (i, ihigh) in tqdm.tqdm_notebook( 
        enumerate(ihigh_range), total=len(ihigh_range) ):

    q_threshold.Q = False
    q_threshold.ilow = .9*ihigh
    q_threshold.ihigh = ihigh

    sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=.1, dense_output=True )   
    (S, I, R, D, Tq) = sol.y
    total_qtime[i] = Tq[-1]
    final_mortality[i] = D[-1] / P0*100

threshold.ihigh_range = ihigh_range
threshold.total_qtime = total_qtime
threshold.final_mortality = final_mortality

In [ ]:
figsize( 12, 4)
subplot( 1, 2, 1)
plot( threshold.ihigh_range, threshold.total_qtime, '.-' )

subplot( 1, 2, 2 )
plot( threshold.ihigh_range, threshold.final_mortality, '.-' )

In [ ]:
figsize(6, 4)
plot( threshold.total_qtime, threshold.final_mortality, '.-' )
xlabel('Days in quarantine')
ylabel('Eventual death toll (%)')

In [ ]:
# Continuous quarantines, 3 days each.
ihigh_range = logspace( -2, log10(.2), num=30 )
#ihigh_range = linspace( .01, .2, num=30 )
total_qtime = empty_like(ihigh_range)
final_mortality = empty_like(ihigh_range)

SIR.quarantine_status = q_intermittent
q_intermittent.len = 3
for (i, ihigh) in tqdm.tqdm_notebook( 
        enumerate(ihigh_range), total=len(ihigh_range) ):
    
    q_intermittent.Q = False
    q_intermittent.ihigh = ihigh

    sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=.1, dense_output=True )   
    (S, I, R, D, Tq) = sol.y
    total_qtime[i] = Tq[-1]
    final_mortality[i] = D[-1] / P0*100

intermittent = types.SimpleNamespace()

intermittent.ihigh_range = ihigh_range
intermittent.total_qtime = total_qtime
intermittent.final_mortality = final_mortality
intermittent.len = q_intermittent.len

In [ ]:
figsize(6, 4)
plot( intermittent.total_qtime, intermittent.final_mortality, '.-' )
xlabel('Length of quarantine (days)')
ylabel('Total mortalities (%)')

In [ ]:
# Find when to start a contiguous quarantine of a month to minimzie mortality
SIR.quarantine_status = q_interval
period.length = 180
q_start_range = arange( 60, 101, step=1.0 )
final_mortality = empty_like(q_start_range)
period.interval = types.SimpleNamespace()

for i, q_start in enumerate( tqdm.tqdm_notebook( q_start_range, total=len(q_start_range) )):
    q_interval.start = q_start
    q_interval.end = q_interval.start + period.length
    
    sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=1, dense_output=False )
    (S, I, R, D, Tq) = sol.y
    tt = sol.t
    final_mortality[i] = D[-1]
    
    if not hasattr( period.interval, 'D' ) or final_mortality[i] < period.interval.D[-1]:
        # Min attained
        period.interval = types.SimpleNamespace(
            start=q_interval.start, end=q_interval.end, qtime=Tq[-1],
            S=S, I=I, R=R, D=D, tt=tt )

In [ ]:
figsize( 6, 4 )
plot( q_start_range, final_mortality, '.-' )
title( f'Min attained by starting on day {period.interval.start}')

In [ ]:
# Delayed quarantines

#qtime_range = logspace( 0, log10(365), num=30 )
#qtime_range = linspace( 0, 360, num=30, endpoint=False )
qtime_range = concatenate((
    arange( 3, 45, step=3.0 ),
    arange( 45, 120, step=15.0 ),
    arange( 120, 390, step=30.0)
))
total_qtime = empty_like(qtime_range)
start_day = zeros_like( qtime_range )
final_mortality = full_like(qtime_range, 100)

SIR.quarantine_status = q_interval
for (i, qtime) in tqdm.tqdm_notebook( 
        enumerate(qtime_range), total=len(qtime_range) ):
    
    # Figure out when to start the quarantine. Try between days 60 and 90
    for start in arange( 60, 90, step=1.0 ):
        q_interval.start = start
        q_interval.end = q_interval.start + qtime

        # Low resolution run to check
        sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=1.0, dense_output=False )
        (S, I, R, D, Tq) = sol.y
        if D[-1]/P0*100 < final_mortality[i]:
            start_day[i] = start
            final_mortality[i] = D[-1]/P0*100

    # Now do a higher resolution run
    q_interval.start = start_day[i]
    q_interval.end = q_interval.start + qtime
    sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=0.1, dense_output=True )
    
    (S, I, R, D, Tq) = sol.y
    total_qtime[i] = Tq[-1]
    final_mortality[i] = D[-1] / P0*100

delayed = types.SimpleNamespace(
    qtime_range=copy(qtime_range), start_day=copy(start_day), total_qtime=copy(total_qtime), 
    final_mortality=copy(final_mortality) )

In [ ]:
# Initial quarantine
# Reuse qtime_range / etc from delayed quarantines
SIR.quarantine_status = q_interval
for (i, qtime) in tqdm.tqdm_notebook( 
        enumerate(qtime_range), total=len(qtime_range) ):
    
    q_interval.start = 0
    q_interval.end = q_interval.start + qtime
    sol = solve_ivp( SIR, [0, T_max], array([S0, I0, R_0, D0, Tq0]),
                 max_step=0.1, dense_output=True )
    
    (S, I, R, D, Tq) = sol.y
    total_qtime[i] = Tq[-1]
    final_mortality[i] = D[-1] / P0*100
    
atstart = types.SimpleNamespace(
    qtime_range=copy(qtime_range), start_day=0, total_qtime=copy(total_qtime), 
    final_mortality=copy(final_mortality) )

In [ ]:
figsize(6, 4)

plot( atstart.total_qtime, atstart.final_mortality, '.-',
    label='Quarantine imposed initially')
plot( delayed.total_qtime, delayed.final_mortality, '.-',
    label='Quarantine delayed')
plot( intermittent.total_qtime, intermittent.final_mortality, '.-',
    label=f'Intermittent quarantine' )
xlabel('Days in quarantine')
ylabel('Fatalities (%) after 18 months')
legend()

In [ ]:
plot( delayed.total_qtime, delayed.start_day, '.-'  )

In [ ]:
atstart.final_mortality[0], atstart.qtime_range[0]

In [ ]:
delayed.final_mortality[9], delayed.qtime_range[9], delayed.start_day[9]